# Fixed detector priors (step1)
The SnowStorm detector priors were not applied in the earlier fits: NNMFit only reads the key `prior`, the config used `priors`.
The fits were re-run with the fixed key (`..._fixedprior`). Here: the previous HESE results, the fit without detector priors
(PassingHESE + _fine combined, as in the earlier `fit_result.ipynb`) and the fit with fixed priors, all SAY; the new SAY vs Poisson
fit; and the 2D (γ, Φ0) scan compared with HESE-7.5.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import NNMFit

sys.path.insert(0, "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step2_hese_flavor_reunblind")
from analysis_utils import (scan_errors, CombinedScan, combine_2d_scans, load_hese75_2d, PREVIOUS_RESULTS,
                            load_triangle_scan, combine_scans, flavor_fractions, latex_comparison_table, PARAMETERS)

plotting_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step1_hese_flux_reunblind/studies/plots_compare_fixedprior"
os.makedirs(plotting_path, exist_ok=True)

In [ ]:
import matplotlib.font_manager as font_manager
font_axis_label = {'family': 'serif',
        'color':  'black',
        'weight': 'normal',
        'size': 22,
        }
font_title = {'family': 'serif',
        'color':  'black',
        'weight': 'bold',
        'size': 20,
        }
font_legend = font_manager.FontProperties(family='serif',
                                   weight='normal',
                                   style='normal', size=13)

plt.rcParams["figure.figsize"] = (6,6)
plt.rcParams.update({'font.family':'serif'})

## 1D scans

In [ ]:
data_fits = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux_try2/data_fits"
scan = "nbestfit50_all_param_2sigma_1D_10steps_PassingHESE"
drop_unsuccessful = False  # as in fit_result.ipynb
parameters = [p for p in PARAMETERS if p not in ("astro_nue_ratio", "astro_nutau_ratio")]

def load(name):
    return NNMFit.utilities.ScanHandler(f"{data_fits}/{name}", dump=False, drop_unsuccessful=drop_unsuccessful)

# without detector priors: PassingHESE + finer scan combined
wrong_scans = {name: load(name) for name in [scan, f"{scan}_fine"]}
scan_dict = {"wrong": CombinedScan(wrong_scans, {n: s.get_freefit(drop_unsuccessful="True") for n, s in wrong_scans.items()})}
bestfits = {"wrong": scan_dict["wrong"].bestfit}
for name, scan_name in [("SAY", f"{scan}_fixedprior"), ("Poisson", f"{scan}_Poisson_fixedprior")]:
    scan_dict[name] = load(scan_name)
    bestfits[name] = scan_dict[name].get_freefit(drop_unsuccessful="True")

errors = {name: scan_errors(scan_dict[name], bestfits[name], parameters, drop_unsuccessful) for name in scan_dict}
# prompt_norm: profile stays below 1 down to the physical bound 0 -> lower error = distance to 0 (as in fit_result.ipynb)
for name in errors:
    bf, lo, hi = errors[name]["prompt_norm"]
    if np.isnan(lo):
        errors[name]["prompt_norm"] = (bf, bf, hi)

for name, bf in bestfits.items():
    print(f"{name}: freefit llh = {bf['llh']:.4f}")

In [ ]:
def show(columns, parameters):
    df = pd.DataFrame({(name, col): {p: (v[i] if v is not None and v[i] is not None else np.nan) for p, v in res.items()}
                       for name, res in columns.items() for i, col in enumerate(["bestfit", "-1sig", "+1sig"])})
    with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 250, "display.max_columns", 30):
        display(df.reindex(parameters))

## SAY: previous HESE results, without detector priors, with fixed priors

In [ ]:
say_columns = {
    "HESE-7.5":             PREVIOUS_RESULTS["HESE-7.5"],
    "HESE-12 flavor":       PREVIOUS_RESULTS["HESE-12 flavor"],
    "no detector priors":   errors["wrong"],
    "fixed priors":         errors["SAY"],
}
show(say_columns, parameters)
print(latex_comparison_table(say_columns,
                             caption=r"Step1 best fit values with $1\sigma$ asymmetric uncertainties (SAY): previous HESE results, "
                                     r"this analysis without the detector priors, and with the detector priors applied.",
                             label="tab:step1_fixedprior", parameters=parameters))

## Fixed priors: SAY vs Poisson

In [ ]:
likelihood_columns = {"SAY": errors["SAY"], "Poisson": errors["Poisson"]}
show(likelihood_columns, parameters)
print(latex_comparison_table(likelihood_columns,
                             caption=r"Step1 best fit values with $1\sigma$ asymmetric uncertainties with the detector priors applied, "
                                     r"for the SAY and the Poisson likelihood.",
                             label="tab:step1_fixedprior_SAY_Poisson", parameters=parameters))

## 2D scan (γ, Φ0)
Without detector priors: the three rounds of the 2D scan combined (lowest llh per grid point, as in `fit_result_2D.ipynb`).

In [ ]:
data_fits_2D = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux_try2/data_fits_2D"
scan_2d = "nbestfit20_gamma_norm_larger_range_2D_225steps"
x_param, y_param = "gamma_astro", "astro_norm"

def load_2d(names):
    """combined 2D scan of several rounds: (x, y, dllh, bestfit), dllh relative to the lowest freefit llh"""
    scans = {n: NNMFit.utilities.ScanHandler(f"{data_fits_2D}/{n}", dump=False, drop_unsuccessful=True) for n in names}
    freefits = {n: s.get_freefit(drop_unsuccessful=True) for n, s in scans.items()}
    best = min(freefits, key=lambda n: freefits[n]["llh"])
    xyd = [scans[n].get_scan_xydllh(x=x_param, y=y_param, dllh_ref=freefits[best]["llh"], interpolate=True, drop_unsuccessful=True)
           for n in names]
    x, y, dllh = combine_2d_scans(*zip(*xyd))
    return x, y, dllh, freefits[best]

scans_2d = {
    "no detector priors": load_2d([scan_2d, f"{scan_2d}_round2", f"{scan_2d}_round3"]),
    "fixed priors":       load_2d([f"{scan_2d}_fixedprior"]),
}
G_hese, N_hese, Z_hese, hese_levels, hese_bf = load_hese75_2d()
for name, (x, y, dllh, bf) in scans_2d.items():
    print(f"{name}: gamma = {bf[x_param]:.3f}, phi0 = {bf[y_param]:.3f}, llh = {bf['llh']:.4f}, NaN grid points: {np.isnan(dllh).sum()}")

In [ ]:
def plot_2d(names, colors, savepath):
    lw, linestyles, our_levels = 3.0, ["-", "--"], [2.27, 5.99]  # 68%, 95% (as in fit_result_2D.ipynb)
    fig, ax = plt.subplots(1, 1)
    lh, ll = [], []
    for name, color in zip(names, colors):
        if name == "HESE-7.5":
            ax.contour(G_hese, N_hese, Z_hese, levels=hese_levels, colors=color, linewidths=lw, linestyles=linestyles)
            bfx, bfy = hese_bf
        else:
            x, y, dllh, bf = scans_2d[name]
            ax.contour(x, y, dllh, levels=our_levels, colors=color, linewidths=lw, linestyles=linestyles)
            bfx, bfy = bf[x_param], bf[y_param]
        ax.plot([bfx], [bfy], "*", color=color, markersize=12)
        lh.append(Line2D([0], [0], color=color, linewidth=lw, marker="*", markersize=10))
        ll.append(f"{name}: " + r"$\gamma$ = " + f"{bfx:.2f}, " + r"$\phi_0$ = " + f"{bfy:.2f}")
    lh += [Line2D([0], [0], color="grey", linestyle=ls, linewidth=2) for ls in linestyles]
    ll += ["68%", "95%"]
    ax.set_xlabel(r"$\gamma$")
    ax.set_ylabel(r"$\Phi_{0,\mu}$")
    ax.legend(lh, ll, loc="upper left", frameon=True, fancybox=True, shadow=True, fontsize=10)
    fig.savefig(savepath, bbox_inches="tight")
    fig.savefig(savepath.replace(".pdf", ".png"), bbox_inches="tight", dpi=100)
    plt.show()

plot_2d(["fixed priors", "HESE-7.5"], ["black", "C0"], f"{plotting_path}/contour_fixedprior_vs_hese7.5.pdf")
plot_2d(["HESE-7.5", "no detector priors", "fixed priors"], ["C0", "C3", "black"], f"{plotting_path}/contour_hese7.5_wrong_fixedprior.pdf")